# 4 pamoka – Įrankių naudojimo dizaino šablonas

Šioje pamokoje sužinosite apie AI agentams skirtą **įrankių naudojimo** dizaino šabloną, naudojant Microsoft Agent Framework (Python). Apžvelgsime:

- Funkcinių įrankių apibrėžimą naudojant `@tool` dekoratorių ir tipizuotus parametrus
- Įrankių schemų pateikimą, kad modelis žinotų, ką kiekvienas įrankis atlieka
- Įrankių vykdymo valdymą su `approval_mode`
- **Struktūruoto išvesties** grąžinimą per Pydantic modelius ir `response_format`

Scenarijus – **kelionių užsakymo agentas**, galintis ieškoti krypčių, tikrinti galimus laikus ir gauti skrydžių informaciją.


## Sąranka


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Įrankių apibrėžimas naudojant @tool dekoratorių

`@tool` dekoratorius paverčia paprastą Python funkciją į įrankį, kurį agentas gali iškviesti.
Pagrindinės mintys:

- **docstring** tampa įrankio aprašu, kurį mato modelis.
- **Tipų anotacijos** (įskaitant `Annotated` su aprašymais) apibrėžia įrankio schemą.
- `approval_mode` nulemia, ar vartotojas turi patvirtinti kiekvieną iškvietimą prieš jį vykdant.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Agentas su keliais įrankiais

Perduokite visus tris įrankius klientui, kad modelis galėtų iškviesti tuos, kurių reikia atsakant į vartotojo klausimą.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Struktūruotas išvestis su įrankiais

Nustatant `response_format` į Pydantic modelį, agentas priverčiamas grąžinti gerai tipizuotą JSON objektą, o ne laisvos formos tekstą. Tai naudinga, kai žemutinės grandies kodas turi programiškai apdoroti rezultatą.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Įrankių patvirtinimo šablonai

Parametras `approval_mode` objekte `@tool` kontroliuoja, ar įrankio iškvietimai reikalauja žmogaus patvirtinimo prieš vykdant:

| Režimas | Elgsena |
|---|---|
| `"never_require"` | Įrankis vykdomas automatiškai – nereikia naudotojo patvirtinimo. |
| `"always_require"` | Kiekvienas iškvietimas privalo būti patvirtintas naudotojo prieš vykdymą. |

Naudokite `"always_require"` įrankiams, kurie turi šalutinių poveikių (pvz., skrydžio užsakymas, kreditinės kortelės nurašymas), kad žmogus išliktų procese. 


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Santrauka

Šiame pamokoje išmokote, kaip:

1. **Apibrėžti įrankius** naudojant `@tool` dekoratorių su tipizuotais parametrais ir dokumentacijos eilutėmis, kurios veikia kaip įrankio schema.
2. **Sudėti kelis įrankius** taip, kad agentas galėtų juos iškvietinėti eilės tvarka, atsakant į sudėtingus užklausimus.
3. **Gauti struktūruotą išvestį** perduodant Pydantic modelį kaip `response_format`.
4. **Valdyti įrankio patvirtinimą** su `approval_mode`, kad jautrių operacijų atveju būtų užtikrinta žmogaus kontrolė.

Šie modeliai sudaro pagrindą kuriant patikimus, gamybai paruoštus agentus, kurie gali saugiai sąveikauti su išorinėmis sistemomis.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
